# 🚀 FactChkBR — Demo Direto no VS Code (Kernel Google Colab)

Para rodar na máquina do Google Colab sem sair do VS Code:
1. No canto superior direito deste notebook no VS Code, clique em **Select Kernel** → **Google Colab** (ou **Colab**).
2. Clique em **Run All** (ou execute as 3 células abaixo em sequência).
3. **Zero Google Drive**: os datasets (`datasets/*.tsv`), o classificador ONNX (`v1.0.0`) e o motor `JevK5-Lite` são baixados automaticamente na sessão.

In [ ]:
# 1️⃣ Clonar/atualizar repositórios e instalar dependências na VM do Colab
import os
import subprocess
import sys

WORKDIR = "/content" if os.path.exists("/content") else os.path.abspath(".")
os.chdir(WORKDIR)

repos = {
    "factchkbr_backend": "https://github.com/Residencia-em-IA-Grupo-6/factchkbr_backend.git",
    "telegram_bot": "https://github.com/Residencia-em-IA-Grupo-6/telegram_bot.git",
}

for name, url in repos.items():
    if not os.path.exists(name):
        print(f"📦 Clonando {name}...")
        subprocess.run(["git", "clone", url], check=True)
    else:
        print(f"🔄 Sincronizando {name}...")
        subprocess.run(["git", "-C", name, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", name, "reset", "--hard", "origin/main"], check=True)

print("⚙️ Instalando dependências (pode levar ~1 min na 1ª vez)...")
for req in ["factchkbr_backend/requirements.txt", "telegram_bot/requirements.txt"]:
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", req],
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        print(res.stdout)
        print(res.stderr)
        raise RuntimeError(f"Falha ao instalar {req}")

print("✅ Ambiente pronto!")

In [ ]:
# 2️⃣ Configurar credenciais (.env) para Backend e Bot
import getpass
import os

def get_secret(key_name: str, prompt_text: str) -> str:
    # 1. Tenta variável de ambiente já definida
    if os.environ.get(key_name):
        return os.environ[key_name]
    # 2. Tenta Secrets do Google Colab
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val:
            return val
    except Exception:
        pass
    # 3. Solicita no input do VS Code (oculto)
    return getpass.getpass(f"{prompt_text}: ").strip()

GROQ_API_KEY = get_secret("GROQ_API_KEY", "Insira sua GROQ_API_KEY")
TELEGRAM_BOT_TOKEN = get_secret("TELEGRAM_BOT_TOKEN", "Insira seu TELEGRAM_BOT_TOKEN")

with open("factchkbr_backend/.env", "w", encoding="utf-8") as f:
    f.write(
        f"LLM_PROVIDER=groq\n"
        f"GROQ_API_KEY={GROQ_API_KEY}\n"
        f"GROQ_MODEL=openai/gpt-oss-120b\n"
        f"TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}\n"
        f"HEALTH_ONLY_MODE=true\n"
    )

with open("telegram_bot/.env", "w", encoding="utf-8") as f:
    f.write(
        f"TELEGRAM_BOT_TOKEN={TELEGRAM_BOT_TOKEN}\n"
        f"BACKEND_URL=http://127.0.0.1:8000\n"
    )

print("✅ Arquivos .env gerados com sucesso!")

In [ ]:
# 3️⃣ Pré-aquecer modelos (ONNX + JevK5-Lite), subir Backend e iniciar o Bot do Telegram
import subprocess
import sys
import time
import urllib.request

print("🧠 Baixando e aquecendo modelos (ONNX INT8 + JevK5-Lite) na primeira execução...")
subprocess.run(
    [
        sys.executable,
        "-c",
        "from app.services.onnx_classifier import get_onnx_classifier; "
        "from app.services.decision_classifier import get_decision_classifier; "
        "from app.services.factcheck_corpus import get_factcheck_corpus; "
        "get_onnx_classifier(); get_decision_classifier(preload=True); get_factcheck_corpus(); "
        "print('✅ Modelos e Corpus carregados!')",
    ],
    cwd="factchkbr_backend",
    check=True,
)

# Encerra instância anterior do backend caso a célula seja reexecutada
subprocess.run(["pkill", "-f", "uvicorn app.main:app"], check=False)

backend_log = open("backend.log", "w", encoding="utf-8")
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="factchkbr_backend",
    stdout=backend_log,
    stderr=subprocess.STDOUT,
)

print("⏳ Aguardando API FastAPI ficar online em http://127.0.0.1:8000/health ...")
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as resp:
            if resp.status == 200:
                print("🚀 Backend online e pronto!")
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Backend não respondeu a tempo. Verifique backend.log.")

print("🤖 Iniciando Bot do Telegram (Streaming SSE ativo)... Envie sua mensagem no Telegram!")
try:
    subprocess.run([sys.executable, "bot.py"], cwd="telegram_bot", check=True)
except KeyboardInterrupt:
    print("🛑 Demo encerrada pelo usuário.")
finally:
    backend_proc.terminate()